# L1 · Edge benchmark of the U-Net: ONNX and INT8 on a CPU (no GPU)

Task **L1** from `Essential_task_delegation/Next_Steps_Roadmap.md`, following Tamanna's Plan 1 (`Tamanna_task_2/Plan_1`). Goal **O4**: a measured speed, memory and accuracy table for the segmenter.

**What it does**
1. **Exports R9** (U-Net + ResNet-50, the D4 champion) to ONNX FP32 with a fixed 1024 × 1024 input; every validation image is 1024 × 1024.
2. **Quantises it to INT8 in two ways:**
   - **dynamic:** weights only, no data needed;
   - **static:** QDQ format, per-channel weights, activation ranges calibrated on 100 *training* images.
3. **Compares four variants on the same CPU:** PyTorch FP32, ONNX FP32, ONNX INT8 dynamic and ONNX INT8 static. For each:
   - file size;
   - RAM used while running (process memory growth, approximate);
   - milliseconds per image: the median of 3 timed runs per image after a warm-up, on 50 validation images;
   - mIoU and Destroyed IoU on all 645 validation images (standard prediction, no TTA), and the share of pixels where it agrees with PyTorch FP32.

**Where it runs:** a Kaggle **CPU** session (4 Xeon vCPUs), so no GPU quota is used.
- The accuracy numbers hold on any CPU.
- The milliseconds describe Kaggle's cloud CPU, not a field laptop. The INT8-over-FP32 speedup is the part that carries over. Laptop timings can be added later as a short run on a PC.

**How to run on Kaggle**
1. Settings: **CPU** (no accelerator), internet ON (Cell 1 installs onnx, onnxruntime, onnxscript and segmentation-models-pytorch 0.3.4).
2. Inputs: the DisasterM3 mirror (attached automatically). Checkpoint: the `HF_TOKEN` secret (R9's private repo), or a dataset holding `R9/final_model.pth`.
3. First run with `SMOKE_TEST = True` (20 validation images, about 15 minutes). Then set it to `False` and use Save & Run All.
   - The full run is about 3–5 hours on 4 vCPUs: 4 variants × 645 images at 1024 px.
   - CPU sessions don't use the weekly GPU quota.
4. Outputs in `/kaggle/working/L1/`:
   - `edge_benchmark.csv`, `edge_benchmark.png`, `side_by_side.png`;
   - the three `.onnx` files;
   - `config.json`, `README.md` (with the CPU model).

**Done when:** all four rows are measured on the same machine. INT8 counts as usable if Destroyed IoU drops by less than 0.01 against PyTorch FP32.

In [1]:
# ── Cell 1: Install the export and quantisation tools ──────────────────────
import importlib.metadata as _md
import subprocess
import sys

PINNED = {"segmentation-models-pytorch": "0.3.4"}     # same version as the D4 notebooks (checkpoint key layout)
EXTRA = ["onnx", "onnxruntime", "onnxscript"]           # onnxscript: needed by the newer torch ONNX exporter

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

missing = [f"{p}=={v}" for p, v in PINNED.items() if _version(p) != v] + [p for p in EXTRA if _version(p) is None]
if missing:
    print("⏳ Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import onnx
import onnxruntime as ort
import torch
print(f"✓ torch {torch.__version__}, onnx {onnx.__version__}, onnxruntime {ort.__version__}, "
      f"python {sys.version.split()[0]}")

⏳ Installing: segmentation-models-pytorch==0.3.4, onnxruntime, onnxscript
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 2.0 MB/s eta 0:00:00
✓ torch 2.11.0+cpu, onnx 1.22.0, onnxruntime 1.31.0, python 3.13.15


In [2]:
# ── Cell 2: Configuration, checkpoint and the machine ─────────────────────
import glob
import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import psutil
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/L1")

NUM_CLASSES = 4
SMOKE_TEST = False             # True = 20 validation, 10 calibration and 5 latency images
MODEL_NAME, HF_USER = "R9", "AbrarAlam"
CKPT_PATH = None               # set a path to skip the search, e.g. "/kaggle/input/my-ckpts/R9/final_model.pth"
INPUT_SIZE = 1024              # every validation image is 1024 x 1024 (checked); the ONNX input is fixed
OPSET = 17
N_CALIBRATION = 100            # training images for static INT8 calibration
N_LATENCY = 50                 # validation images for the latency runs
LATENCY_REPEATS = 3            # timed runs per image after a warm-up; the median is kept
EXPECTED = (0.5373, 0.2921)    # R9's D4 CSV, standard prediction: (mIoU, Destroyed IoU)

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """Order: CKPT_PATH, then a /kaggle/input folder named R9 / d4_R9 / R9-results, then Hugging Face."""
    if CKPT_PATH:
        return CKPT_PATH
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo, errors = f"{HF_USER}/disasterm3-d4-{name}", []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}.")

def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or platform.machine()

CKPT = find_checkpoint(MODEL_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
THREADS = os.cpu_count()
torch.set_num_threads(THREADS)
MACHINE = {"cpu": cpu_name(), "threads": THREADS, "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
           "os": platform.platform(), "torch_threads": torch.get_num_threads()}
print(f"✓ L1 edge benchmark{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_NAME} from {CKPT}")
print(f"  Machine: {MACHINE}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ L1 edge benchmark: R9 from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9/snapshots/34897e1aa70742cac62a379561e1f969dccd4e10/final_model.pth
  Machine: {'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'threads': 4, 'ram_gb': 33.7, 'os': 'Linux-6.18.48+-x86_64-with-glibc2.39', 'torch_threads': 4}


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run), calibration images and preprocessing ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
calib_pairs = random.Random(0).sample(train_pairs, N_CALIBRATION)   # training images only, never validation
if SMOKE_TEST:
    val_pairs, calib_pairs = val_pairs[:20], calib_pairs[:10]
latency_pairs = val_pairs[:5 if SMOKE_TEST else N_LATENCY]

# The D4 notebooks' A.Normalize (ImageNet mean and std, max_pixel_value 255), without albumentations
MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def preprocess(path):
    """Image file -> float32 array of shape 1 x 3 x H x W, normalised exactly as in training."""
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB).astype(np.float32)
    return np.ascontiguousarray(((image - MEAN) / STD).transpose(2, 0, 1)[None])

sizes = Counter(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE).shape for p in val_pairs)
assert set(sizes) == {(INPUT_SIZE, INPUT_SIZE)}, f"The fixed ONNX input needs {INPUT_SIZE} px images, found {sizes}"
print(f"✓ Validation images: {len(val_pairs)} (expected 645 when SMOKE_TEST is False); "
      f"calibration: {len(calib_pairs)} training images; latency: {len(latency_pairs)} images")

✓ Validation images: 645 (expected 645 when SMOKE_TEST is False); calibration: 100 training images; latency: 50 images


In [ ]:
# ── Cell 5: Export to ONNX FP32, then quantise to INT8 (dynamic and static) ──
import segmentation_models_pytorch as smp
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType, quantize_dynamic,
                                      quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process

def load_pytorch():
    """D4 checkpoints: U-Net + ResNet-50, 4 classes. Accepts a state dict or a resume checkpoint."""
    state = torch.load(CKPT, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.eval()

PATHS = {"ONNX FP32": OUT_DIR / "r9_fp32.onnx",
         "ONNX INT8 dynamic": OUT_DIR / "r9_int8_dynamic.onnx",
         "ONNX INT8 static": OUT_DIR / "r9_int8_static.onnx"}
prepared = OUT_DIR / "r9_fp32_prepared.onnx"     # shape-inferred and optimised copy that the quantisers start from

t0 = time.time()
model = load_pytorch()
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
export_args = dict(input_names=["image"], output_names=["logits"], opset_version=OPSET)
try:
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=False, **export_args)
except Exception as e:                            # newer torch may drop the classic exporter
    print(f"Classic exporter failed ({type(e).__name__}: {str(e)[:100]}); using the dynamo exporter")
    torch.onnx.export(model, (dummy,), str(PATHS["ONNX FP32"]), dynamo=True, **export_args)
onnx.checker.check_model(str(PATHS["ONNX FP32"]))
del model
quant_pre_process(str(PATHS["ONNX FP32"]), str(prepared))

quantize_dynamic(str(prepared), str(PATHS["ONNX INT8 dynamic"]), weight_type=QuantType.QInt8)

class TrainingImages(CalibrationDataReader):
    """Feeds the calibration images one by one, so static INT8 learns realistic activation ranges."""
    def __init__(self, pairs):
        self.items = iter(pairs)
    def get_next(self):
        p = next(self.items, None)
        return None if p is None else {"image": preprocess(p["image_path"])}

# The calibrator reduces every tensor to its min and max inside the graph, so 100 images need little RAM.
# Don't set extra_options={"CalibMaxIntermediateOutputs": ...}: in onnxruntime 1.31 it clears the collected
# data without merging it, and calibration then fails with "No data is collected" (the first L1 run, 18 Oct).
quantize_static(str(prepared), str(PATHS["ONNX INT8 static"]), TrainingImages(calib_pairs),
                quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)

FILE_MB = {"PyTorch FP32": os.path.getsize(CKPT) / 1e6, **{k: os.path.getsize(p) / 1e6 for k, p in PATHS.items()}}
print(f"✓ Export and quantisation done in {(time.time() - t0) / 60:.1f} min")
for k, mb in FILE_MB.items():
    print(f"  {k:18s} {mb:7.1f} MB")

## Measure: accuracy on all validation images, then speed
- **Variants:** each one is loaded fresh, so its RAM figure (growth of this process's memory while it runs) includes its weights.
- **Accuracy:** the standard prediction (no TTA) and the dataset-level confusion matrix, as in the D4 notebooks. PyTorch FP32 should land close to R9's D4 CSV (0.5373 / 0.2921). That run used fp16 autocast on a GPU, so expect small differences in the 4th decimal.
- **Speed:** a warm-up, then 3 timed runs per image on the first 50 validation images. The median per image is kept, and the table reports the median and 95th percentile over images.

In [ ]:
# ── Cell 6: Accuracy of every variant on the validation images ─────────────
import gc
import pandas as pd
from tqdm.auto import tqdm

VARIANTS = ["PyTorch FP32", "ONNX FP32", "ONNX INT8 dynamic", "ONNX INT8 static"]

def make_runner(kind):
    """Image array (1 x 3 x H x W) -> logits array, for one variant."""
    if kind == "PyTorch FP32":
        model = load_pytorch()
        def run(x):
            with torch.inference_mode():
                return model(torch.from_numpy(x)).numpy()
        return run
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    session = ort.InferenceSession(str(PATHS[kind]), options, providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]

def confusion(gt, pred):
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def iou_from(cm):
    tp = np.diag(cm).astype(np.float64)
    return tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)

process = psutil.Process()
# PyTorch's predictions, kept for the agreement check; filled now so their memory isn't charged to a variant
reference = np.full((len(val_pairs), INPUT_SIZE, INPUT_SIZE), 255, np.uint8)
examples, accuracy = {}, {}
t0 = time.time()
for kind in VARIANTS:
    gc.collect()
    rss_before = process.memory_info().rss
    run = make_runner(kind)
    peak, cm, agree, total = process.memory_info().rss, np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64), 0, 0
    for i, p in enumerate(tqdm(val_pairs, desc=kind)):
        pred = run(preprocess(p["image_path"]))[0].argmax(0).astype(np.uint8)
        cm += confusion(cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE), pred)
        if kind == "PyTorch FP32":
            reference[i] = pred
        else:
            agree += int((pred == reference[i]).sum())
            total += pred.size
        if i == 0:
            examples[kind] = pred
        peak = max(peak, process.memory_info().rss)
    iou = iou_from(cm)
    accuracy[kind] = {"miou": float(iou.mean()), "destroyed_iou": float(iou[3]),
                      "agreement_with_pytorch": 1.0 if kind == "PyTorch FP32" else agree / total,
                      "ram_mb": (peak - rss_before) / 1e6}
    print(f"  {kind:18s} mIoU {iou.mean():.4f} | Destroyed IoU {iou[3]:.4f} | "
          f"agrees with PyTorch on {accuracy[kind]['agreement_with_pytorch']:.2%} of pixels | "
          f"({(time.time() - t0) / 60:.0f} min so far)")
    del run
    gc.collect()

d4_miou, d4_destroyed = EXPECTED
pt = accuracy["PyTorch FP32"]
if not SMOKE_TEST:
    close = abs(pt["miou"] - d4_miou) < 2e-3 and abs(pt["destroyed_iou"] - d4_destroyed) < 2e-3
    print(f"\nPyTorch FP32 vs R9's D4 CSV ({d4_miou} / {d4_destroyed}): {'matches' if close else 'DIFFERS: check the checkpoint'}")

In [ ]:
# ── Cell 7: Speed of every variant (median of 3 timed runs per image, after a warm-up) ──
inputs = [preprocess(p["image_path"]) for p in latency_pairs]
latency = {}
for kind in VARIANTS:
    run = make_runner(kind)
    run(inputs[0])                                   # warm-up (first call allocates and optimises)
    per_image = []
    for x in tqdm(inputs, desc=kind):
        times = []
        for _ in range(LATENCY_REPEATS):
            t = time.perf_counter()
            run(x)
            times.append(time.perf_counter() - t)
        per_image.append(1000 * float(np.median(times)))
    latency[kind] = {"ms_median": float(np.median(per_image)), "ms_p95": float(np.percentile(per_image, 95))}
    print(f"  {kind:18s} {latency[kind]['ms_median']:8.0f} ms per image (median), "
          f"p95 {latency[kind]['ms_p95']:.0f} ms")
    del run
    gc.collect()

In [ ]:
# ── Cell 8: The benchmark table, plots, README and config ─────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

base_ms = latency["PyTorch FP32"]["ms_median"]
rows = []
for kind in VARIANTS:
    a, l = accuracy[kind], latency[kind]
    rows.append({"variant": kind, "file_mb": round(FILE_MB[kind], 1), "ram_mb": round(a["ram_mb"]),
                 "ms_per_image": round(l["ms_median"]), "ms_p95": round(l["ms_p95"]),
                 "speedup_vs_pytorch": round(base_ms / l["ms_median"], 2),
                 "miou": round(a["miou"], 4), "destroyed_iou": round(a["destroyed_iou"], 4),
                 "destroyed_drop": round(pt["destroyed_iou"] - a["destroyed_iou"], 4),
                 "pixel_agreement_with_pytorch": round(a["agreement_with_pytorch"], 4)})
table = pd.DataFrame(rows)
table["int8_usable"] = [("yes" if r["destroyed_drop"] < 0.01 else "no") if "INT8" in r["variant"] else ""
                        for _, r in table.iterrows()]
table.to_csv(OUT_DIR / "edge_benchmark.csv", index=False)
print(table.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
short = [v.replace("ONNX ", "").replace("PyTorch ", "PyTorch\n") for v in table["variant"]]
axes[0].bar(short, table["file_mb"], color="#4a7ab8")
axes[0].set_ylabel("File size (MB)")
axes[1].bar(short, table["ms_per_image"], color="#c0613c")
axes[1].set_ylabel("ms per 1024 px image (median)")
for _, r in table.iterrows():
    axes[2].scatter(r["ms_per_image"], r["destroyed_iou"], s=70)
    axes[2].annotate(r["variant"], (r["ms_per_image"], r["destroyed_iou"]), fontsize=8,
                     xytext=(5, 4), textcoords="offset points")
axes[2].set_xlabel("ms per image")
axes[2].set_ylabel("Destroyed IoU")
fig.suptitle(f"R9 on {MACHINE['cpu']} ({MACHINE['threads']} threads)")
fig.tight_layout()
fig.savefig(OUT_DIR / "edge_benchmark.png", dpi=110)
plt.close(fig)

# The same validation image from every variant (0 background, 1 intact, 2 damaged, 3 destroyed)
colours = np.array([[0, 0, 0], [60, 180, 75], [255, 200, 0], [230, 25, 75]], np.uint8)
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(4.2 * len(VARIANTS), 4.4))
for ax, kind in zip(axes, VARIANTS):
    ax.imshow(colours[examples[kind]])
    ax.set_title(kind)
    ax.axis("off")
fig.suptitle(Path(val_pairs[0]["image_path"]).name)
fig.tight_layout()
fig.savefig(OUT_DIR / "side_by_side.png", dpi=90)
plt.close(fig)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

prepared.unlink(missing_ok=True)     # an intermediate copy, not one of the four variants
config = {"task": "L1", "smoke_test": SMOKE_TEST, "model": MODEL_NAME, "checkpoint": str(CKPT), "machine": MACHINE,
          "input_size": INPUT_SIZE, "opset": OPSET, "calibration_images": len(calib_pairs),
          "validation_images": len(val_pairs), "latency_images": len(latency_pairs),
          "latency_repeats": LATENCY_REPEATS, "onnxruntime": ort.__version__, "torch": torch.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L1: edge benchmark of the R9 U-Net", "",
    f"Machine: {MACHINE['cpu']}, {MACHINE['threads']} threads, {MACHINE['ram_gb']} GB RAM ({MACHINE['os']}). "
    f"onnxruntime {ort.__version__}, torch {torch.__version__}.", "",
    f"Accuracy on {len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''} (standard prediction, "
    f"no TTA); speed as the median of {LATENCY_REPEATS} timed runs per image on {len(latency_pairs)} images. "
    "RAM is the growth of the process memory while the variant ran (approximate).", "",
    as_markdown(table), "",
    "INT8 counts as usable if Destroyed IoU drops by less than 0.01 against PyTorch FP32.", "",
    "![Benchmark](edge_benchmark.png)", "", "![Side by side](side_by_side.png)", "",
    "Files: `edge_benchmark.csv`, the three `.onnx` models, `config.json`.",
]), encoding="utf-8")
print(f"✓ Wrote edge_benchmark.csv, the plots, README.md and config.json to {OUT_DIR}")